
# 🧑‍💻 K-Nearest Neighbors (KNN) Workshop

Welcome to the **KNN Active Learning Workshop**!  
In this session, you will learn how to apply the **K-Nearest Neighbors algorithm** in practice, while also exploring **data analysis techniques** and **machine learning design patterns**.

This workshop is designed for **peer programming** and **active learning** — you will collaborate with classmates to brainstorm, code, and reflect.

---


**Team:** Sultan Atanda (9114837) · John [student ID] · Antonio Sainz [student ID] · Eche [student ID]

**Our dataset:** hourly weather for Waterloo, ON in 2025, pulled from the Open-Meteo API and stored in Neon PostgreSQL. We use KNN to classify each hour as **Clear, Cloudy, Rain or Snow**.

| Part | What we did |
|---|---|
| Part 1 | Iris warm-up, plus the effect of *k* and scaling |
| Part 2 | `DataLoader` (API → Neon), `Preprocessor`, `KNNModel` and `Evaluator` |
| Part 3 | `KNNPipeline` chains the four steps, then results, error analysis and validation checks |
| Reflection | Answers to the five questions and the peer comparison |

---


## 🎯 Learning Objectives

By the end of this workshop, you should be able to:

1. Explain the intuition behind the K-Nearest Neighbors algorithm.  
2. Acquire and preprocess datasets from different sources (CSV, APIs, Relational Databases).  
3. Apply KNN for classification on multiple datasets.  
4. Use the **Machine Learning Pipeline Pattern** to architect modular, reproducible ML code.  
5. Reflect on KNN’s performance, strengths, and limitations, and when to apply it in real-world scenarios.  

---



## ⏳ Before the Workshop

Make sure you are ready with the following setup:

1. **Environment Setup**  
   - Use Python 3.9+ with `scikit-learn`, `pandas`, `matplotlib`, `requests`, and `sqlalchemy`.  
   - You can install everything with:  
     ```bash
     pip install scikit-learn pandas matplotlib requests sqlalchemy psycopg2-binary
     ```

2. **Data Sources**  
   You will use **three types of data sources** in this workshop:
   - **CSV file**: e.g., [UCI ML Repository](https://archive.ics.uci.edu/) datasets.  
   - **API**: e.g., the [OpenWeather API](https://open-meteo.com/) for weather data.  
   - **Relational Database**: either connect to a public database or create your own using [Neon](https://neon.com/).  

   👉 Bring at least one dataset of your choice from *each category*.  

3. **Peer Programming**  
   - Pair up with another student.  
   - Take turns: one codes, one reviews. Switch every 15 minutes.  


## 🧩 Explore the KNN Algorithm with a Simulator

Before coding, let’s explore how KNN works visually. This simulator allows you to place data points on a 2D plane, adjust the number of neighbors (*k*), and observe how predictions change based on distance and class boundaries. Experimenting here builds intuition before we implement KNN in Python.

🔗 [Interactive KNN Simulator](https://k-nearest-neighbors.vercel.app/)

---



## 🌸 Part 1: Warm-up with the Iris Dataset

We’ll begin with the **Iris dataset**, a classic dataset in machine learning.

**Task:** Implement and evaluate a KNN classifier to predict the species of iris flowers.



In [1]:

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Load dataset
iris = load_iris()
X, y = iris.data, iris.target

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create and train your KNN classifier
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)

# Evaluate
y_pred = knn.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))


Accuracy: 1.0


Now that we’ve trained and evaluated a KNN classifier on the Iris dataset, let’s move from raw accuracy metrics to a more visual understanding. By plotting the first two features—**sepal length** and **sepal width**—we can observe how different iris species cluster, and how KNN relies on these dimensions for classification.


In [2]:
import pandas as pd
import plotly.express as px

iris_df = pd.DataFrame(X, columns=iris.feature_names)
iris_df["species"] = iris.target_names[y]

fig = px.scatter(iris_df, x=iris.feature_names[0], y=iris.feature_names[1], color="species",
                 title="Iris Dataset - First Two Features", template="plotly_white")
fig.update_traces(marker=dict(size=9, line=dict(width=1, color="black")))
fig.show()

The scatter plot gave us a visual sense of how the Iris data clusters when using sepal length and width. To move beyond visualization, we now use our trained KNN model to make predictions on unseen test data, evaluate its accuracy, and inspect example classifications in detail.  

💡 **Note for reflection:** The performance of KNN depends heavily on two design choices:  
1. The value of *k* (number of neighbors).  
2. How features are scaled (since distances drive the algorithm).  

Think about how changing either might alter your results.


In [3]:
# Predict the species for the test set
y_pred = knn.predict(X_test)

# Evaluate the accuracy
accuracy = accuracy_score(y_test, y_pred)
print("KNN Classifier for Iris Dataset")
sample_idx = 10
print("Sample features:", X_test[sample_idx])
print("True label:", y_test[sample_idx])
print("Predicted label:", y_pred[sample_idx])
#print("Test data sample:", X_test)
#print("Test data predictions:", y_pred)
print(f"Test set accuracy: {accuracy:.2f}")

# Explanation:
# KNN classifies a new data point by looking at the 'k' closest labeled data points in the training set.
# The majority label among these neighbors is assigned to the new point.
# In this example, we use KNN to classify iris flowers based on their features.

KNN Classifier for Iris Dataset
Sample features: [6.5 3.2 5.1 2. ]
True label: 2
Predicted label: 2
Test set accuracy: 1.00


By comparing predicted labels against the ground truth, we can directly measure how well the model generalizes beyond the training set. Accuracy provides a simple performance metric, but it is equally valuable to look at individual predictions. This allows us to interpret success cases and identify where the model struggles.

In [4]:
predicted_label = y_pred[sample_idx]

fig = px.scatter(iris_df, x=iris.feature_names[0], y=iris.feature_names[1], color="species",
                 title="Iris Dataset with New Data Point Highlighted", template="plotly_white")
fig.update_traces(marker=dict(size=9, line=dict(width=1, color="black")))
fig.add_scatter(x=[X_test[sample_idx, 0]], y=[X_test[sample_idx, 1]], mode="markers+text",
                marker=dict(symbol="star", size=22, color="gold", line=dict(width=1, color="black")),
                text=[f"Predicted: {iris.target_names[predicted_label]}"], textposition="middle right",
                name="New Data Point")
fig.show()

#### Effect of *k* and scaling on Iris

To answer the reflection note above, we compare 5-fold cross-validated accuracy for several values of *k*, with and without `StandardScaler`.

In [5]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rows = []
for k in [1, 3, 5, 7, 15, 30, 60]:
    unscaled = cross_val_score(KNeighborsClassifier(n_neighbors=k), X, y, cv=5).mean()
    scaled = cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)), X, y, cv=5).mean()
    rows.append({"k": k, "unscaled": unscaled, "scaled": scaled})

pd.DataFrame(rows).set_index("k").round(3)

,unscaled,scaled
k,,
1,0.960,0.947
3,0.967,0.953
5,0.973,0.960
7,0.980,0.960
15,0.967,0.940
30,0.940,0.927
60,0.887,0.853


**What this shows:** accuracy is stable from k = 1 to 15 (about 0.95–0.98) and falls once k gets large (0.887 unscaled at k = 60), because each prediction starts averaging over flowers from other species (each species only has 50). Scaling doesn't help here and is slightly worse, because all four Iris features are already lengths in centimetres. Our weather data is different: pressure is around 960 hPa while hour runs from 0 to 23, so scaling matters a lot there.


## 📊 Part 2: Apply KNN on a Real-World Dataset

Now it’s your turn to **go beyond the Iris dataset**.

### Step 1: Acquire Data
Choose one of the following:  
- **CSV**: Download a dataset (e.g., Titanic dataset from Kaggle or UCI).  
- **API**: Query an API like OpenWeather or a sports stats API. Use `requests` to fetch JSON data.  
- **Database**: Connect to a Neon-hosted PostgreSQL database.  

👉 Work with your partner to **select a real-world dataset** that excites you.  

### Step 2: Preprocess Data
- Clean missing values.  
- Normalize numerical features.  
- Encode categorical variables if needed.  

### Step 3: Implement KNN
- Split into train/test sets.  
- Fit KNN.  
- Evaluate accuracy, precision, recall, or other metrics as appropriate.  



### Our dataset

| | |
|---|---|
| **Source** | [Open-Meteo Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api), stored in our **Neon PostgreSQL** database (table `knn_weather`) |
| **Location / period** | Waterloo, ON (43.46°N, 80.52°W), every hour of 2025 (8,760 rows) |
| **Target** | `condition`: **Clear**, **Cloudy**, **Rain** or **Snow**, grouped from the WMO `weather_code` |
| **Features** | temperature, relative humidity, dew point, apparent temperature, surface pressure, cloud cover, wind speed, wind direction, wind gusts, hour of day, month |

We picked this because it's local, real and large enough for KNN, and the target is a real categorical label with four classes. It also uses two of the three source types from the workshop: the API for collection and a relational database for storage.

The `DataLoader` pulls the year from the API (and caches it as a CSV), writes it to Neon if the table is empty, then reads it back from Neon. If `DATABASE_URL` isn't set, or Neon can't be reached, it falls back to the cached CSV so the notebook still runs.

In [6]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import psycopg2
import requests
from dotenv import load_dotenv
from IPython.display import display
from psycopg2.extras import execute_values
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import (StratifiedGroupKFold, StratifiedKFold,
                                     cross_val_score, train_test_split)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

load_dotenv(Path(".env"))
RANDOM_STATE = 42
DATA_PATH = Path("data/waterloo_weather_2025.csv")

#### Step 1: Acquire data (`DataLoader`)

In [7]:
class DataLoader:
    API_URL = "https://archive-api.open-meteo.com/v1/archive"
    TABLE = "knn_weather"
    COLUMNS = [
        "time", "temperature_2m", "relative_humidity_2m", "dew_point_2m",
        "apparent_temperature", "precipitation", "rain", "snowfall", "weather_code",
        "surface_pressure", "cloud_cover", "wind_speed_10m", "wind_direction_10m",
        "wind_gusts_10m",
    ]

    def __init__(self, csv_path=DATA_PATH, latitude=43.4643, longitude=-80.5204, year=2025):
        self.csv_path = Path(csv_path)
        self.latitude = latitude
        self.longitude = longitude
        self.year = year
        self.database_url = os.getenv("DATABASE_URL")
        self.source = None

    def fetch_from_api(self):
        params = {
            "latitude": self.latitude,
            "longitude": self.longitude,
            "start_date": f"{self.year}-01-01",
            "end_date": f"{self.year}-12-31",
            "hourly": self.COLUMNS[1:],
            "timezone": "America/Toronto",
        }
        response = requests.get(self.API_URL, params=params, timeout=60)
        response.raise_for_status()
        df = pd.DataFrame(response.json()["hourly"])
        df["time"] = pd.to_datetime(df["time"])
        self.csv_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(self.csv_path, index=False)
        return df[self.COLUMNS]

    def read_csv(self):
        return pd.read_csv(self.csv_path, parse_dates=["time"])[self.COLUMNS]

    def save_to_database(self, df):
        column_types = ", ".join(
            f"{c} INTEGER" if c == "weather_code" else f"{c} DOUBLE PRECISION"
            for c in self.COLUMNS[1:]
        )
        rows = list(zip(df["time"].dt.strftime("%Y-%m-%d %H:%M:%S"),
                        *(df[c].tolist() for c in self.COLUMNS[1:])))
        conn = psycopg2.connect(self.database_url, connect_timeout=10)
        try:
            with conn, conn.cursor() as cur:
                cur.execute(f"CREATE TABLE IF NOT EXISTS {self.TABLE} (time TIMESTAMP PRIMARY KEY, {column_types})")
                cur.execute(f"SELECT COUNT(*) FROM {self.TABLE}")
                if cur.fetchone()[0] == 0:
                    execute_values(cur, f"INSERT INTO {self.TABLE} ({', '.join(self.COLUMNS)}) VALUES %s", rows)
        finally:
            conn.close()

    def load_from_database(self):
        conn = psycopg2.connect(self.database_url, connect_timeout=10)
        try:
            with conn.cursor() as cur:
                cur.execute(f"SELECT {', '.join(self.COLUMNS)} FROM {self.TABLE} ORDER BY time")
                df = pd.DataFrame(cur.fetchall(), columns=self.COLUMNS)
        finally:
            conn.close()
        df["time"] = pd.to_datetime(df["time"])
        return df

    def load(self, refresh=False):
        raw = self.fetch_from_api() if refresh or not self.csv_path.exists() else self.read_csv()
        if self.database_url:
            try:
                self.save_to_database(raw)
                self.source = f"Neon table {self.TABLE}"
                return self.load_from_database()
            except psycopg2.Error as exc:
                print("Could not reach Neon, using the local copy instead:", exc)
        self.source = str(self.csv_path)
        return raw

In [8]:
loader = DataLoader()
weather = loader.load()

print(f"Loaded {len(weather):,} rows x {weather.shape[1]} columns from {loader.source}")
print(f"Period: {weather['time'].min()} to {weather['time'].max()}")
weather.head()

Loaded 8,760 rows x 14 columns from Neon table knn_weather
Period: 2025-01-01 00:00:00 to 2025-12-31 23:00:00


,time,temperature_2m,relative_humidity_2m,dew_point_2m,apparent_temperature,precipitation,rain,snowfall,weather_code,surface_pressure,cloud_cover,wind_speed_10m,wind_direction_10m,wind_gusts_10m
0,2025-01-01 00:00:00,0.7,96.0,0.2,-3.8,1.0,0.1,0.63,73,962.1,100.0,16.3,6.0,32.4
1,2025-01-01 01:00:00,0.6,96.0,0.1,-4.1,1.0,0.0,0.70,73,962.2,100.0,17.6,354.0,32.4
2,2025-01-01 02:00:00,0.6,95.0,-0.1,-4.4,0.8,0.1,0.49,73,962.4,100.0,19.6,343.0,36.0
3,2025-01-01 03:00:00,0.6,94.0,-0.3,-4.7,0.7,0.1,0.42,73,962.7,100.0,20.9,339.0,38.9
4,2025-01-01 04:00:00,0.6,94.0,-0.4,-4.9,0.3,0.0,0.21,73,963.3,100.0,22.2,334.0,41.4


#### Step 2: Preprocess data (`Preprocessor`)

Decisions:

- **Target:** `weather_code` is grouped into four classes. Codes 0–1 are Clear, 2–3 are Cloudy, 51–67 and 80–82 are Rain, and 71–77 and 85–86 are Snow. Any other code (e.g. 45/48, fog) doesn't fit a class and is dropped.
- **Leakage:** `weather_code`, `rain`, `snowfall` and `precipitation` are removed from the features. The label is built from them, so keeping them would let the model read the answer.
- **Feature engineering:** `hour` and `month` are extracted from `time`, because the chance of snow or clear skies depends on season and time of day.
- **Missing values:** there are none in the 2025 data, but a median imputer stays in the pipeline in case a future API pull has gaps.
- **Scaling:** KNN uses distances, so unscaled pressure (around 960 hPa) would outweigh humidity (0–100) and hour (0–23). `StandardScaler` puts every feature on the same scale.
- **No leakage through preprocessing:** the imputer and scaler are fitted on the **training set only** and then applied to the test set.
- **Encoding:** there are no categorical input features (`timezone` is a constant text column and isn't used). The target stays as text labels, which scikit-learn's KNN accepts directly.
- **Split:** 80/20, stratified on the target so the rarer Rain and Snow hours appear in the same proportion in both sets.

In [9]:
class Preprocessor:
    LEAKAGE_COLUMNS = ["weather_code", "rain", "snowfall", "precipitation"]
    FEATURES = [
        "temperature_2m", "relative_humidity_2m", "dew_point_2m", "apparent_temperature",
        "surface_pressure", "cloud_cover", "wind_speed_10m", "wind_direction_10m",
        "wind_gusts_10m", "hour", "month",
    ]
    CLASSES = ["Clear", "Cloudy", "Rain", "Snow"]

    def __init__(self, features=None):
        self.features = features or self.FEATURES
        self.imputer = SimpleImputer(strategy="median")
        self.scaler = StandardScaler()

    @staticmethod
    def to_condition(code):
        if pd.isna(code):
            return np.nan
        code = int(code)
        if code <= 1:
            return "Clear"
        if code <= 3:
            return "Cloudy"
        if 51 <= code <= 67 or 80 <= code <= 82:
            return "Rain"
        if 71 <= code <= 77 or 85 <= code <= 86:
            return "Snow"
        return np.nan

    def clean(self, df):
        data = df.drop_duplicates(subset="time").copy()
        data["time"] = pd.to_datetime(data["time"])
        data["condition"] = data["weather_code"].map(self.to_condition)
        data["hour"] = data["time"].dt.hour
        data["month"] = data["time"].dt.month
        data = data.dropna(subset=["condition"]).reset_index(drop=True)
        self.rows_dropped = len(df) - len(data)
        self.time = data["time"]
        return data[self.features], data["condition"]

    def split(self, X, y, test_size=0.2):
        return train_test_split(X, y, test_size=test_size, stratify=y, random_state=RANDOM_STATE)

    def fit_transform(self, X_train):
        return self.scaler.fit_transform(self.imputer.fit_transform(X_train))

    def transform(self, X):
        return self.scaler.transform(self.imputer.transform(X))

In [10]:
preprocessor = Preprocessor()
X, y = preprocessor.clean(weather)

print("Missing values in features:", int(X.isna().sum().sum()))
print("Rows dropped (no matching class or duplicate hour):", preprocessor.rows_dropped)
print("Features used:", list(X.columns))

class_counts = (y.value_counts().reindex(Preprocessor.CLASSES)
                .rename_axis("condition").reset_index(name="hours"))
class_counts["share"] = (class_counts["hours"] / class_counts["hours"].sum() * 100).round(1)
display(class_counts)

px.bar(class_counts, x="condition", y="hours", text="share", color="condition",
       title="Hours per weather condition (2025)", template="plotly_white"
       ).update_traces(texttemplate="%{text}%").update_layout(showlegend=False).show()

Missing values in features: 0
Rows dropped (no matching class or duplicate hour): 0
Features used: ['temperature_2m', 'relative_humidity_2m', 'dew_point_2m', 'apparent_temperature', 'surface_pressure', 'cloud_cover', 'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m', 'hour', 'month']


,condition,hours,share
0,Clear,3079,35.1
1,Cloudy,3950,45.1
2,Rain,900,10.3
3,Snow,831,9.5


**What this shows:** the classes are imbalanced. Cloudy (45%) and Clear (35%) make up 80% of the hours, while Rain (10%) and Snow (9.5%) are much rarer. A model that always said "Cloudy" would already be right 45% of the time. That's why we stratify the split and judge the model mainly by **macro F1** rather than accuracy.

#### Step 3: Implement KNN (`KNNModel` and `Evaluator`)

`KNNModel` wraps scikit-learn's `KNeighborsClassifier`. Before the final fit, it tunes **k (3–25)** and the **distance metric (Euclidean or Manhattan)** using 5-fold stratified cross-validation on the **training set only**, scored by macro F1. The test set isn't touched until the final evaluation.

`Evaluator` reports accuracy plus macro precision, recall and F1. We use macro averages because the classes are imbalanced: macro F1 gives Rain and Snow the same weight as Cloudy. `Evaluator` also compares the model against a majority-class baseline and draws the charts.

In [11]:
class KNNModel:
    def __init__(self, k_values=range(3, 26), metrics=("euclidean", "manhattan"), cv_folds=5):
        self.k_values = k_values
        self.metrics = metrics
        self.cv_folds = cv_folds
        self.best_k = None
        self.best_metric = None
        self.model = None

    def tune(self, X_train, y_train):
        cv = StratifiedKFold(n_splits=self.cv_folds, shuffle=True, random_state=RANDOM_STATE)
        rows = []
        for metric in self.metrics:
            for k in self.k_values:
                scores = cross_val_score(KNeighborsClassifier(n_neighbors=k, metric=metric),
                                         X_train, y_train, cv=cv, scoring="f1_macro", n_jobs=-1)
                rows.append({"k": k, "metric": metric, "cv_mean": scores.mean(), "cv_std": scores.std()})
        self.tuning_results = pd.DataFrame(rows)
        best = self.tuning_results.loc[self.tuning_results["cv_mean"].idxmax()]
        self.best_k, self.best_metric = int(best["k"]), best["metric"]
        return self.tuning_results

    def fit(self, X_train, y_train):
        self.model = KNeighborsClassifier(n_neighbors=self.best_k or 5,
                                          metric=self.best_metric or "euclidean")
        self.model.fit(X_train, y_train)
        return self

    def predict(self, X):
        return self.model.predict(X)


class Evaluator:
    def __init__(self, labels=None):
        self.labels = labels

    @staticmethod
    def metrics(y_true, y_pred):
        return {
            "accuracy": accuracy_score(y_true, y_pred),
            "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        }

    def baseline(self, y_train, y_test):
        dummy = DummyClassifier(strategy="most_frequent").fit(np.zeros((len(y_train), 1)), y_train)
        return self.metrics(y_test, dummy.predict(np.zeros((len(y_test), 1))))

    @staticmethod
    def compare(results):
        return pd.DataFrame(results).T.round(3)

    def report(self, y_true, y_pred):
        report = classification_report(y_true, y_pred, labels=self.labels,
                                       output_dict=True, zero_division=0)
        return pd.DataFrame(report).T.round(3)

    def plot_k_tuning(self, tuning, best_k, best_metric):
        fig = go.Figure()
        for metric, part in tuning.groupby("metric"):
            fig.add_trace(go.Scatter(x=part["k"], y=part["cv_mean"], mode="lines+markers", name=metric,
                                     error_y=dict(type="data", array=part["cv_std"], thickness=1)))
        best = tuning[(tuning["k"] == best_k) & (tuning["metric"] == best_metric)].iloc[0]
        fig.add_trace(go.Scatter(x=[best_k], y=[best["cv_mean"]], mode="markers", name=f"best (k={best_k})",
                                 marker=dict(size=16, symbol="star", line=dict(width=1, color="black"))))
        fig.update_layout(title="Choosing k: cross-validated macro F1 (training set)",
                          xaxis_title="k (number of neighbours)", yaxis_title="Mean macro F1 (5-fold CV)",
                          legend_title="Distance", template="plotly_white")
        return fig

    def plot_comparison(self, comparison):
        long = comparison.reset_index(names="model").melt(id_vars="model", var_name="metric", value_name="score")
        return px.bar(long, x="metric", y="score", color="model", barmode="group", text_auto=".2f",
                      range_y=[0, 1], title="KNN vs majority-class baseline (test set)",
                      template="plotly_white")

    def plot_confusion(self, y_true, y_pred):
        cm = confusion_matrix(y_true, y_pred, labels=self.labels)
        return px.imshow(cm, x=self.labels, y=self.labels, text_auto=True, color_continuous_scale="Blues",
                         labels=dict(x="Predicted", y="Actual", color="Hours"),
                         title="Confusion matrix (test set)", template="plotly_white")


## 🏗️ Part 3: Architecting with the ML Pipeline Pattern

In professional ML, we use the **Machine Learning Pipeline Pattern**:  

- Chain modular components in a **linear sequence** or **DAG structure**.  
- Each step should be reusable, testable, and clearly defined.  

Example pipeline for KNN:  
1. **Data Acquisition** → from CSV/API/DB  
2. **Data Cleaning & Preprocessing** → normalization, encoding, etc.  
3. **Feature Engineering**  
4. **Model Training**  
5. **Evaluation & Validation**  
6. **Deployment / Reflection**  

👉 Your task: Implement your experiment using **functions or classes** for each stage.  


### Our pipeline

`KNNPipeline` chains the four components in order:

`DataLoader` → `Preprocessor` → `KNNModel` → `Evaluator`

Each component does one job and can be swapped or reused on its own. Section 3.4 reuses the same classes with a different feature list, and section 3.6 reuses them on Iris.

In [12]:
class KNNPipeline:
    def __init__(self, loader, preprocessor, model, evaluator):
        self.loader = loader
        self.preprocessor = preprocessor
        self.model = model
        self.evaluator = evaluator

    def run(self):
        self.raw = self.loader.load()
        self.X, self.y = self.preprocessor.clean(self.raw)
        self.X_train, self.X_test, self.y_train, self.y_test = self.preprocessor.split(self.X, self.y)
        X_train_scaled = self.preprocessor.fit_transform(self.X_train)
        X_test_scaled = self.preprocessor.transform(self.X_test)
        self.tuning = self.model.tune(X_train_scaled, self.y_train)
        self.model.fit(X_train_scaled, self.y_train)
        self.y_pred = self.model.predict(X_test_scaled)
        self.results = self.evaluator.compare({
            "Majority baseline": self.evaluator.baseline(self.y_train, self.y_test),
            f"KNN (k={self.model.best_k}, {self.model.best_metric})": self.evaluator.metrics(self.y_test, self.y_pred),
        })
        return self

In [13]:
pipeline = KNNPipeline(DataLoader(), Preprocessor(), KNNModel(), Evaluator(labels=Preprocessor.CLASSES)).run()

print(f"Data source: {pipeline.loader.source}")
print(f"Train: {len(pipeline.X_train):,} hours | Test: {len(pipeline.X_test):,} hours")
print(f"Best k = {pipeline.model.best_k}, distance = {pipeline.model.best_metric}")

Data source: Neon table knn_weather
Train: 7,008 hours | Test: 1,752 hours
Best k = 3, distance = euclidean


#### 3.1 Choosing k

In [14]:
evaluator = pipeline.evaluator
evaluator.plot_k_tuning(pipeline.tuning, pipeline.model.best_k, pipeline.model.best_metric).show()
pipeline.tuning.sort_values("cv_mean", ascending=False).head(5).round(3)

,k,metric,cv_mean,cv_std
0,3,euclidean,0.767,0.016
23,3,manhattan,0.765,0.009
2,5,euclidean,0.750,0.012
25,5,manhattan,0.748,0.005
4,7,euclidean,0.743,0.011


**What this shows:** macro F1 is highest at **k = 3** (0.767) and falls steadily as k grows. A bigger k averages over more neighbours, which drowns out the small Rain and Snow classes in favour of Cloudy. Euclidean and Manhattan are almost tied (0.767 vs 0.765), so k matters much more than the distance metric. Section 3.5 explains why such a small k wins on this data.

#### 3.2 Test-set performance

In [15]:
display(pipeline.results)
evaluator.plot_comparison(pipeline.results).show()

,accuracy,precision_macro,recall_macro,f1_macro
Majority baseline,0.451,0.113,0.250,0.155
"KNN (k=3, euclidean)",0.867,0.823,0.789,0.804


**What this shows:** on the 1,752 held-out hours, KNN reaches **86.7% accuracy and 0.804 macro F1**, compared with 45.1% and 0.155 for always predicting Cloudy. Macro recall (0.789) is lower than accuracy because the model does better on the common classes than on the rare ones. The per-class table below shows where.

#### 3.3 Where the model makes mistakes

In [16]:
evaluator.report(pipeline.y_test, pipeline.y_pred)

,precision,recall,f1-score,support
Clear,0.939,0.969,0.954,616.000
Cloudy,0.855,0.882,0.869,790.000
Rain,0.723,0.622,0.669,180.000
Snow,0.774,0.681,0.724,166.000
accuracy,0.867,0.867,0.867,0.867
macro avg,0.823,0.789,0.804,1752.000
weighted avg,0.863,0.867,0.864,1752.000


**What this shows:** **Clear** (F1 0.954) and **Cloudy** (0.869) are predicted very well. **Rain** (0.669) and **Snow** (0.724) are harder: with under 1,000 examples each, a new rainy or snowy hour often has more Cloudy neighbours than Rain or Snow ones. Rain recall is the weakest number at 0.622, so about 4 in 10 rainy hours are missed.

In [17]:
evaluator.plot_confusion(pipeline.y_test, pipeline.y_pred).show()

**What this shows:** almost every mistake involves **Cloudy**. 57 rainy hours and 51 snowy hours were labelled Cloudy, and 65 cloudy hours were labelled Rain or Snow. That makes physical sense: an overcast hour just before or after precipitation has almost the same temperature, humidity and pressure. Rain and Snow are almost never confused with each other (2 hours in total), because temperature separates them cleanly.

#### 3.4 How much does `cloud_cover` carry the model?

Open-Meteo decides Clear vs Cloudy (codes 0–3) mostly from cloud cover. To see how much the model depends on that one feature, we rerun the same pipeline without it.

In [18]:
no_cloud = [f for f in Preprocessor.FEATURES if f != "cloud_cover"]
ablation = KNNPipeline(DataLoader(), Preprocessor(features=no_cloud), KNNModel(),
                       Evaluator(labels=Preprocessor.CLASSES)).run()

display(Evaluator.compare({
    "All features": Evaluator.metrics(pipeline.y_test, pipeline.y_pred),
    "Without cloud_cover": Evaluator.metrics(ablation.y_test, ablation.y_pred),
}))
pd.DataFrame({
    "All features": evaluator.report(pipeline.y_test, pipeline.y_pred)["f1-score"],
    "Without cloud_cover": evaluator.report(ablation.y_test, ablation.y_pred)["f1-score"],
}).loc[Preprocessor.CLASSES]

,accuracy,precision_macro,recall_macro,f1_macro
All features,0.867,0.823,0.789,0.804
Without cloud_cover,0.749,0.743,0.717,0.729


,All features,Without cloud_cover
Clear,0.954,0.774
Cloudy,0.869,0.752
Rain,0.669,0.651
Snow,0.724,0.739


**What this shows:** without `cloud_cover`, accuracy falls from 0.867 to 0.749 and macro F1 from 0.804 to 0.729. Nearly all of the loss is in **Clear** (F1 0.954 → 0.774) and **Cloudy** (0.869 → 0.752), while Rain and Snow barely move. So `cloud_cover` is the feature that separates sunny from overcast hours. It's a real measured variable, not leakage, but it explains why Clear vs Cloudy is so easy for the model.

#### 3.5 Are neighbouring hours leaking between train and test?

Weather changes slowly, so 2 PM and 3 PM on the same day are almost the same row. A random split puts many of these "twin" hours on opposite sides, and KNN can then find a near-copy of each test hour in the training data. To check this, we cross-validate again but keep **whole weeks together**, so each week is either entirely in training or entirely in testing.

In [19]:
weeks = (pipeline.preprocessor.time.dt.dayofyear - 1) // 7
rows = []
for k in [3, 7, 15, 25]:
    knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k, metric=pipeline.model.best_metric))
    random_cv = cross_val_score(knn, pipeline.X, pipeline.y, scoring="f1_macro",
                                cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE))
    week_cv = cross_val_score(knn, pipeline.X, pipeline.y, groups=weeks, scoring="f1_macro",
                              cv=StratifiedGroupKFold(5, shuffle=True, random_state=RANDOM_STATE))
    rows.append({"k": k, "random split": random_cv.mean(), "weeks kept together": week_cv.mean()})

week_check = pd.DataFrame(rows).set_index("k").round(3)
fig = px.line(week_check.reset_index().melt(id_vars="k", var_name="validation", value_name="macro F1"),
              x="k", y="macro F1", color="validation", markers=True, range_y=[0, 1],
              title="Random split vs whole-week split", template="plotly_white")
fig.show()
week_check

,random split,weeks kept together
k,,
3,0.796,0.612
7,0.762,0.618
15,0.721,0.616
25,0.688,0.596


**What this shows:** with a random split, macro F1 is 0.796 at k = 3. When whole weeks are held out, it drops to about **0.61**, and k = 3 no longer wins: every k from 3 to 15 scores around 0.61–0.62. So part of the headline score comes from near-duplicate neighbouring hours landing in both train and test, which is also why the smallest k looked best. **About 0.61 macro F1 is the realistic estimate for a week the model has never seen.** That's still about four times the baseline (0.155). We keep the stratified 80/20 split for the main results, as the workshop asks, and report both.

#### 3.6 Reusing the components on Iris

As a quick check that the components are reusable, `Preprocessor` (split and scaling), `KNNModel` and `Evaluator` run unchanged on the Iris data from Part 1.

In [20]:
iris_prep, iris_model = Preprocessor(), KNNModel(k_values=range(1, 16))
Xi_train, Xi_test, yi_train, yi_test = iris_prep.split(iris.data, iris.target)
iris_model.tune(iris_prep.fit_transform(Xi_train), yi_train)
iris_model.fit(iris_prep.fit_transform(Xi_train), yi_train)
yi_pred = iris_model.predict(iris_prep.transform(Xi_test))

print(f"Iris best k = {iris_model.best_k} ({iris_model.best_metric})")
pd.Series(Evaluator.metrics(yi_test, yi_pred)).round(3)

Iris best k = 3 (euclidean)


accuracy           0.933
precision_macro    0.944
recall_macro       0.933
f1_macro           0.933
dtype: float64

**What this shows:** the same classes run on Iris without any changes, and tuning picks k = 3. Test accuracy is 0.933 (2 of 30 flowers wrong), compared with 1.00 in Part 1. The difference is scaling. All four Iris features are already lengths in centimetres, so standardising them only reweights petal against sepal measurements, as the k/scaling table in Part 1 also showed. On our weather data, where the units differ widely, scaling is essential.

## 🔍 End of Part 1: Reflection

Discuss the following with your peers:  

1. How did KNN perform on your dataset?  
2. What preprocessing decisions most affected performance?  
3. What are the strengths and weaknesses of KNN?  
4. When would you **not** use KNN in practice?  
5. How could this workflow be extended in an **MLOps context** (automation, CI/CD, monitoring)?  

👉 Write your reflections in the cell below.


#### 💭 Reflection: Add Your Talking Points

---

**1. How did KNN perform on our dataset?**

On the stratified 80/20 split, KNN (k = 3, Euclidean) reached **86.7% accuracy and 0.804 macro F1**, compared with 45.1% and 0.155 for a majority-class baseline. Clear and Cloudy were predicted very well (F1 0.95 and 0.87). Rain and Snow were harder (0.67 and 0.72) and were mostly confused with Cloudy. When we held out whole weeks so that near-duplicate hours couldn't appear on both sides, macro F1 fell to about **0.61**. We treat that as the realistic performance on genuinely new days.

**2. Which preprocessing decisions affected performance most?**

- **Removing the leaking columns** (`weather_code`, `rain`, `snowfall`, `precipitation`). With them, the model would score almost perfectly while learning nothing useful.
- **Scaling with `StandardScaler`, fitted on the training set only.** KNN is distance-based, so unscaled pressure would dominate every other feature.
- **Keeping `cloud_cover`.** Removing it cut accuracy from 0.867 to 0.749, almost entirely in Clear vs Cloudy.
- **Stratifying the split and scoring with macro F1**, so the rare Rain and Snow classes are judged fairly.
- **Adding `hour` and `month`** from the timestamp. They're stored as plain numbers, so 23:00 and 00:00 look far apart even though they're an hour apart. Cyclical (sin/cos) encoding would be the next thing to try.

**3. Strengths and weaknesses of KNN**

*Strengths:* it's simple and easy to explain ("this hour looks like these 3 past hours"), it has no real training step, and it handles four classes naturally.

*Weaknesses:* it's very sensitive to feature scaling and to irrelevant features. It's slow at prediction time, because every query is compared against all ~7,000 training hours. It drifts toward the majority class as k grows. And, as section 3.5 showed, near-duplicate rows in time-series data can make it look better than it really is.

**4. When would we not use KNN in practice?**

- Large or real-time datasets that need fast predictions, since every prediction scans the whole training set.
- High-dimensional data, where distances stop being meaningful.
- Problems where missing a rare class is costly, such as a severe-weather warning. Our Rain recall was only 0.62.
- Forecasting future weather. Models that use time order (lag features, gradient boosting, sequence models) fit that better than KNN.

**5. How could this workflow be extended in an MLOps context?**

- **Automation:** schedule the `DataLoader` to pull new days from Open-Meteo into Neon and retrain on a fixed schedule.
- **CI/CD:** run the pipeline in GitHub Actions on every pull request, with a test that fails if macro F1 drops below a set threshold.
- **Experiment tracking:** log k, distance metric, feature list and metrics for each run (e.g. with MLflow).
- **Validation:** use week-grouped validation by default, given section 3.5.
- **Monitoring:** compare live predictions with observed conditions and alert on drift, for example when the season changes.

---

**Peer comparison (with the other team)**

| | Our team | Other team |
|---|---|---|
| Dataset / source | Waterloo hourly weather 2025 (Open-Meteo API → Neon) | |
| Target | Clear / Cloudy / Rain / Snow | |
| Key preprocessing | Leakage columns removed, StandardScaler (train only), hour + month | |
| Best k / distance | 3 / Euclidean | |
| Accuracy / macro F1 | 0.867 / 0.804 | |

*What we learned from their results:*

---

## 🚀 Challenge

Your **challenge is to implement KNN on a brand new dataset** (not Iris).  

- Acquire data from an API or a database.  
- Preprocess it properly.  
- Train and evaluate KNN.  
- Compare results with your peers.  
- **Bonus:** Try tuning hyperparameters (number of neighbors, distance metric).  

### 🤝 Peer Programming Exercise

This workshop is based on **active learning** and requires you to engage in **peer programming** with your classmates.  
Peer programming means that two groups of students work together on the same code:  

- **Driver:** Types the code, explains aloud what they are doing.  
- **Navigator:** Reviews, asks questions, spots mistakes, and thinks strategically about next steps.  

Switch roles every 15–20 minutes.  

### 📝 Step-by-Step Exercise

Work with your partner to complete the following exercise:  

1. **Acquire data**  
   - Choose a dataset from an API (e.g., weather, finance, or sports stats) or connect to a relational database (e.g., using [Neon](https://neon.com/) for PostgreSQL).  
   - Make sure the dataset has at least one categorical label or target variable you can predict.  

2. **Preprocess data**  
   - Handle missing values.  
   - Normalize or scale numerical features.  
   - Encode categorical features as needed.  

3. **Implement KNN using the Pipeline Pattern**  
   - Structure your code into modular functions (or classes) for **data loading, preprocessing, training, and evaluation**.  
   - Chain them together in a clear pipeline.  

4. **Train and evaluate KNN**  
   - Split the dataset into training and test sets.  
   - Fit the KNN model.  
   - Evaluate performance using accuracy, precision, recall, or other suitable metrics.  

5. **Compare results with your peers**  
   - Share your dataset choice, preprocessing decisions, and model results.  
   - Discuss why results may differ across teams.  
   - Reflect on whether KNN was the best choice for your dataset.  

### 📤 Workshop Submission Instructions

To complete the workshop, follow these steps carefully:

1. **Clone the instructor’s repository**  
   ```bash
   git clone https://github.com/ProfEspinosaAIML/KNearestNeighbors_Workshop.git
   cd KNearestNeighbors_Workshop

2. **Create your solution notebook**  
   - Add a new file called: **KNN_Workshop_Solution.ipynb**
   - Implement your code and include Markdown explanations of your thought process, results, and reflections, including the observations and input provided by your peers (the other team working with you)

3. **Add documentation**  
   - Create a README.md file with your name, student ID, and a short summary of your work.

4. **Commit to your local repo**  
   - git add KNN_Workshop_Solution.ipynb README.md
   - git commit -m "Add solution notebook and summary"

5. **Commit to your local repo**  
   - Push to your own remote repo
   - Create a new repository on GitHub called: **KNearestNeighbors_Workshop**
   - Push your code there.

6. **Submit your work**  
   - Create a PDF file with your name and your student ID
   - The URL to your remote repository 
   - Submit the PDF to the dropbox.


---

✅ You’ve now completed the **KNN Active Learning Workshop**.  
You explored multiple data sources, implemented KNN, applied the ML Pipeline Pattern, and reflected on results.

👏 Great work — keep building!  
